# Section 2: Google Cloud's Gen AI Offerings
## GCP Generative AI Leader Certification

This notebook covers hands-on exercises for Google Cloud's generative AI platform:
- Vertex AI Platform overview and SDK usage
- Model Garden: exploring and selecting models
- Vertex AI Studio: interactive prompt design
- RAG implementation with Vertex AI
- Function calling and tool use
- Agent Builder concepts
- Google Search grounding

**Prerequisites**: A GCP project with Vertex AI API enabled and billing configured.

---
## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q google-cloud-aiplatform>=1.60.0 google-generativeai>=0.7.0

In [ ]:
# Authenticate and initialize
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
except ImportError:
    print("Not in Colab. Using default credentials.")

PROJECT_ID = "your-project-id"  # <-- Replace
LOCATION = "us-central1"

import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Vertex AI initialized: project={PROJECT_ID}, location={LOCATION}")

In [ ]:
from vertexai.generative_models import (
    GenerativeModel, GenerationConfig, Part,
    Tool, FunctionDeclaration, grounding
)
print("All imports successful.")

---
## 2. Vertex AI Platform: Core Capabilities

Vertex AI is Google Cloud's unified AI platform. Let's explore its
key capabilities through the SDK.

In [ ]:
# Demonstrate Vertex AI's key generative AI capabilities
try:
    model = GenerativeModel(
        "gemini-1.5-pro",
        system_instruction="You are a Google Cloud solutions architect. "
                          "Provide concise, accurate technical recommendations."
    )

    response = model.generate_content(
        "A retail company wants to build an AI-powered customer service chatbot. "
        "It needs to answer questions from their product catalog and support docs. "
        "Which Google Cloud offerings should they use and why?"
    )
    print("=== Vertex AI Solution Architecture ===")
    print(response.text)

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("=== Vertex AI Solution Architecture ===")
    print("Recommended architecture for retail AI chatbot:")
    print("")
    print("1. **Vertex AI Agent Builder** - Build the conversational agent")
    print("   with dialog flows and natural language understanding.")
    print("2. **Vertex AI Search** - Index product catalog and support docs")
    print("   for RAG-based retrieval with AI-powered summaries.")
    print("3. **Gemini Pro** - Power the generative responses with grounding.")
    print("4. **Cloud Storage / BigQuery** - Store product catalog data.")
    print("5. **IAM + VPC-SC** - Enterprise security controls.")

---
## 3. Model Garden: Choosing the Right Model

Model Garden provides access to Google models, open-source models,
and third-party models. Let's explore how to select the right one.

In [ ]:
# Decision framework: which model for which scenario?
scenarios = [
    {
        "scenario": "High-volume customer support chatbot (10K queries/hour)",
        "best_model": "gemini-1.5-flash",
        "reason": "Fastest response time, lowest cost per token, good quality"
    },
    {
        "scenario": "Complex legal document analysis with reasoning",
        "best_model": "gemini-1.5-pro",
        "reason": "Best reasoning capability, 2M token context for long docs"
    },
    {
        "scenario": "On-premises deployment in air-gapped environment",
        "best_model": "gemma-7b (from Model Garden)",
        "reason": "Open-weight, self-deployable, no API dependency"
    },
    {
        "scenario": "Generate marketing images from text descriptions",
        "best_model": "imagen-3",
        "reason": "Specialized image generation with enterprise safety filters"
    },
    {
        "scenario": "Summarize meeting recordings with transcript",
        "best_model": "gemini-1.5-pro (multimodal)",
        "reason": "Native audio understanding, long context for full meetings"
    },
]

print("=== Model Selection Decision Framework ===")
print(f"{'Scenario':<55} {'Model':<30} {'Reason'}")
print("-" * 120)
for s in scenarios:
    print(f"{s['scenario']:<55} {s['best_model']:<30} {s['reason']}")

---
## 4. Function Calling (Tool Use)

Function calling enables Gemini to interact with external systems.
The model generates structured function call requests that your code executes.

In [ ]:
# Define functions the model can call
try:
    get_product_info = FunctionDeclaration(
        name="get_product_info",
        description="Get product information including price, availability, and description",
        parameters={
            "type": "object",
            "properties": {
                "product_id": {
                    "type": "string",
                    "description": "The product SKU or ID"
                },
                "include_reviews": {
                    "type": "boolean",
                    "description": "Whether to include customer reviews"
                }
            },
            "required": ["product_id"]
        }
    )

    check_inventory = FunctionDeclaration(
        name="check_inventory",
        description="Check real-time inventory for a product at a specific store",
        parameters={
            "type": "object",
            "properties": {
                "product_id": {"type": "string", "description": "Product SKU"},
                "store_id": {"type": "string", "description": "Store location ID"}
            },
            "required": ["product_id", "store_id"]
        }
    )

    # Create a tool with both functions
    retail_tool = Tool(function_declarations=[get_product_info, check_inventory])

    # Model with tools
    model_with_tools = GenerativeModel(
        "gemini-1.5-pro",
        tools=[retail_tool],
        system_instruction="You are a helpful retail assistant."
    )

    response = model_with_tools.generate_content(
        "Is the Nike Air Max 90 (SKU: NKE-AM90-BLK) available at the downtown store (store-001)?"
    )

    # Check if model wants to call a function
    print("=== Function Calling Response ===")
    for candidate in response.candidates:
        for part in candidate.content.parts:
            if hasattr(part, 'function_call') and part.function_call:
                fc = part.function_call
                print(f"Model wants to call: {fc.name}")
                print(f"With arguments: {dict(fc.args)}")
            elif hasattr(part, 'text') and part.text:
                print(f"Text: {part.text}")

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("=== Function Calling Response ===")
    print("Model wants to call: check_inventory")
    print("With arguments: {'product_id': 'NKE-AM90-BLK', 'store_id': 'store-001'}")
    print("")
    print("Key insight: The model DOES NOT execute the function.")
    print("It returns a structured request that YOUR CODE executes.")
    print("This is how agents interact with external APIs and databases.")

---
## 5. Google Search Grounding

Grounding connects Gemini to live Google Search results,
enabling responses about current events and real-time information.

In [ ]:
# Google Search grounding
try:
    model = GenerativeModel("gemini-1.5-pro")

    # Create a Google Search grounding tool
    search_tool = Tool.from_google_search_retrieval(
        grounding.GoogleSearchRetrieval()
    )

    grounded_response = model.generate_content(
        "What are the latest Google Cloud AI announcements from this month?",
        tools=[search_tool]
    )

    print("=== Google Search Grounded Response ===")
    print(grounded_response.text[:500])

    # Check for grounding metadata
    if hasattr(grounded_response, 'candidates'):
        for candidate in grounded_response.candidates:
            if hasattr(candidate, 'grounding_metadata') and candidate.grounding_metadata:
                gm = candidate.grounding_metadata
                print(f"\nGrounding sources: {len(gm.grounding_chunks)} chunks")
                for chunk in gm.grounding_chunks[:3]:
                    if hasattr(chunk, 'web'):
                        print(f"  - {chunk.web.title}: {chunk.web.uri}")

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("=== Google Search Grounded Response ===")
    print("Google Cloud recently announced several AI updates:")
    print("- Gemini 1.5 Pro with 2M token context window")
    print("- New Imagen 3 model for image generation")
    print("- Vertex AI Agent Builder general availability")
    print("- Enhanced RAG capabilities with grounding")
    print("")
    print("Grounding sources: 3 chunks")
    print("  - Google Cloud Blog: cloud.google.com/blog/...")
    print("  - Google Cloud AI: cloud.google.com/ai/...")
    print("")
    print("Key benefit: Search grounding provides CURRENT information")
    print("beyond the model's training data cutoff.")

---
## 6. RAG with Vertex AI

Build a simple RAG pipeline that retrieves context from custom documents
before generating a response.

In [ ]:
# Simplified RAG demonstration
# In production, you'd use Vertex AI Vector Search + RAG API

# Simulated knowledge base (in production: Vertex AI Search or Vector Search)
knowledge_base = {
    "pricing": "Our Enterprise plan costs $30/user/month. Includes Gemini Advanced, "
               "1TB storage, advanced security, and admin controls. Annual billing "
               "gives 20% discount ($24/user/month).",
    "security": "Enterprise plan includes VPC Service Controls, CMEK encryption, "
                "DLP integration, data residency controls, and SOC2/ISO27001 compliance. "
                "Customer data is never used for model training.",
    "features": "Gemini Enterprise includes: Gemini in Gmail, Docs, Sheets, Slides, Meet. "
                "Features: email drafting, document generation, data analysis, presentation "
                "creation, meeting summaries, and NotebookLM for research."
}

def simple_retrieve(query, kb):
    """Simple keyword-based retrieval (production: use embeddings + vector search)"""
    query_lower = query.lower()
    results = []
    for topic, content in kb.items():
        if any(word in query_lower for word in topic.split('_') + [topic]):
            results.append(content)
    if not results:
        results = list(kb.values())  # fallback: return all
    return "\n\n".join(results)

# RAG query
user_question = "How much does the Enterprise plan cost and what security features does it include?"
retrieved_context = simple_retrieve(user_question, knowledge_base)

print("=== RAG Pipeline ===")
print(f"\n1. User Question: {user_question}")
print(f"\n2. Retrieved Context ({len(retrieved_context)} chars):")
print(f"   {retrieved_context[:200]}...")

# Generate grounded response
try:
    model = GenerativeModel("gemini-1.5-pro")
    rag_prompt = f"""Answer the question based ONLY on the provided context.
If the context doesn't contain the answer, say "I don't have that information."

Context:
{retrieved_context}

Question: {user_question}

Answer:"""

    response = model.generate_content(rag_prompt)
    print(f"\n3. Generated Answer:")
    print(response.text)

except Exception as e:
    print(f"\n3. Generated Answer (mock):")
    print("The Enterprise plan costs $30/user/month ($24/user/month with annual billing).")
    print("Security features include: VPC Service Controls, CMEK encryption, DLP")
    print("integration, data residency controls, and SOC2/ISO27001 compliance.")
    print("Importantly, customer data is never used for model training.")

---
## 7. Agent Concepts: Multi-Turn Conversations

Agents maintain state across conversation turns and can use tools.
Let's build a simple multi-turn agent.

In [ ]:
# Multi-turn agent with system instructions
try:
    agent = GenerativeModel(
        "gemini-1.5-pro",
        system_instruction="""You are a Google Cloud solutions consultant.
Help customers choose the right Google Cloud AI offerings.
Ask clarifying questions to understand their needs.
Recommend specific products (Gemini for Workspace, Vertex AI, Agent Builder, etc.).
Be concise and practical."""
    )

    chat = agent.start_chat()

    turns = [
        "We're a mid-size company with 500 employees. We want to use AI to improve productivity.",
        "Mostly office workers who use email, docs, and spreadsheets. No developers on the AI team yet.",
        "What about adding a customer service chatbot later? We have a large FAQ document."
    ]

    print("=== Multi-Turn Agent Conversation ===")
    for turn in turns:
        response = chat.send_message(turn)
        print(f"\nUser: {turn}")
        print(f"Agent: {response.text[:300]}")
        print("-" * 60)

    print(f"\nConversation turns: {len(chat.history)}")

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Conversation ---")
    print("\nUser: We're a mid-size company with 500 employees...")
    print("Agent: I'd recommend starting with **Gemini for Google Workspace**.")
    print("It integrates AI directly into Gmail, Docs, Sheets, and Meet.")
    print("What type of work do most of your employees do?")
    print("-" * 60)
    print("\nUser: Mostly office workers who use email, docs, and spreadsheets...")
    print("Agent: Perfect fit for **Gemini Enterprise** ($30/user/month).")
    print("Key features: email drafting in Gmail, document generation in Docs,")
    print("formula creation in Sheets, meeting summaries in Meet.")
    print("-" * 60)
    print("\nUser: What about adding a customer service chatbot later?")
    print("Agent: For a FAQ-based chatbot, use **Vertex AI Agent Builder**.")
    print("Upload your FAQ docs, and it creates a conversational agent with")
    print("RAG grounding. No developer needed for basic setup.")

---
## 8. Offering Selection Quiz

Test your knowledge of when to recommend which Google Cloud AI offering.

In [ ]:
# Interactive quiz on offering selection
quiz = [
    {
        "q": "A marketing team needs AI to help draft emails and create presentations.",
        "options": ["A) Vertex AI Platform", "B) Gemini for Workspace", "C) Agent Builder", "D) Gemma"],
        "answer": "B",
        "explanation": "Gemini for Workspace integrates AI directly into Gmail (drafting) and Slides (presentations). No-code solution for business users."
    },
    {
        "q": "A dev team wants to build a custom AI app that queries internal databases.",
        "options": ["A) Gemini App", "B) Gemini for Workspace", "C) Vertex AI + Agent Builder", "D) Customer Engagement Suite"],
        "answer": "C",
        "explanation": "Vertex AI + Agent Builder provides the developer platform for custom AI apps with function calling to query databases."
    },
    {
        "q": "A company needs to deploy an LLM in their own on-premises data center.",
        "options": ["A) Gemini Pro API", "B) Vertex AI Endpoints", "C) Gemma from Model Garden", "D) Gemini for Workspace"],
        "answer": "C",
        "explanation": "Gemma is open-weight and can be deployed on any infrastructure, including on-premises. Gemini requires Google's API."
    },
    {
        "q": "A contact center wants to automate Tier 1 support with AI voice agents.",
        "options": ["A) Vertex AI Studio", "B) Customer Engagement Suite", "C) Gemini Advanced", "D) Model Garden"],
        "answer": "B",
        "explanation": "Customer Engagement Suite (formerly CCAI) is purpose-built for contact center automation with virtual agents and agent assist."
    },
    {
        "q": "An enterprise needs Google-quality search over 10M internal documents.",
        "options": ["A) Cloud Search", "B) Vertex AI Search", "C) BigQuery", "D) Gemini App"],
        "answer": "B",
        "explanation": "Vertex AI Search provides enterprise search with AI summaries, extractive answers, and support for millions of documents."
    },
]

print("=== Google Cloud AI Offering Selection Quiz ===")
score = 0
for i, q in enumerate(quiz):
    print(f"\nQ{i+1}: {q['q']}")
    for opt in q['options']:
        print(f"  {opt}")
    print(f"\n  Correct Answer: {q['answer']}")
    print(f"  Explanation: {q['explanation']}")
    print("-" * 60)

print(f"\nStudy these patterns - the exam frequently tests offering selection!")

---
## Summary

In this notebook, you explored:

1. **Vertex AI Platform** - The unified AI development platform for Google Cloud
2. **Model Garden** - Selecting the right model (Google, open-source, third-party)
3. **Function Calling** - Enabling Gemini to interact with external APIs and databases
4. **Google Search Grounding** - Connecting Gemini to live web results
5. **RAG Pipeline** - Retrieving context from custom documents for grounded responses
6. **Multi-Turn Agents** - Building conversational agents with state and system instructions
7. **Offering Selection** - Matching business scenarios to the right Google Cloud AI product

**Key exam takeaways:**
- Vertex AI = developer platform; Gemini for Workspace = end-user product
- Model Garden offers choice: Google, open-source, and third-party models
- Function calling enables agents to interact with external systems
- RAG = custom data grounding; Google Search = public web grounding
- Match the offering to the audience (business user vs developer vs data scientist)

**Next**: [Section 3 - Techniques to Improve Gen AI Model Output](03-improve-model-output.ipynb)